<a href="https://colab.research.google.com/github/AdriaLazaro/btc-cycle-model/blob/main/notebooks/BTC_Cycle_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BTC Cycle Investment Model

Notebook limpio y modular para construir el oscilador de ciclos de Bitcoin.


## Imports y configuracion

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# Dataset y rango historico principal.
DATASET_CANDIDATES = [
    Path("coin-metrics-new-chart.tsv.tsv"),
    Path("../coin-metrics-new-chart.tsv.tsv"),
    Path("/content/coin-metrics-new-chart.tsv.tsv"),
    Path("/content/BTC.tsv.tsv"),
    Path("/content/btc-cycle-model/coin-metrics-new-chart.tsv.tsv"),
    Path("/content/btc-cycle-model-main/coin-metrics-new-chart.tsv.tsv"),
]

START_DATE = "2013-01-01"
FUTURE_END = pd.Timestamp("2031-03-01")

# Rango de drawdown esperado desde el maximo 2025 para estimar el proximo minimo.
DRAWDOWN_SUAVE = 0.66
DRAWDOWN_DURO = 0.76

# Multiplicador minimo -> maximo siguiente para estimar el proximo maximo.
MULT_MIN = 3.0
MULT_MAX = 6.0

# Suavizado semanal del oscilador: 1 conserva todo el detalle real.
ROLLING_SMOOTH = 1


## Funciones reutilizables

In [ ]:
def year_diff(d2, d1):
    """Return the distance between two dates in years."""
    return (pd.Timestamp(d2) - pd.Timestamp(d1)).days / 365.25


def load_btc_data(candidate_paths=None, encodings=("utf-8", "utf-16")):
    """Load BTC price data from several local/Colab paths and print a quick audit."""
    candidate_paths = candidate_paths or DATASET_CANDIDATES
    required_columns = {"Time", "BTC / USD Denominated Closing Price"}
    load_errors = []

    for path in candidate_paths:
        path = Path(path)
        if not path.exists():
            continue

        for encoding in encodings:
            try:
                raw_df = pd.read_csv(path, sep="\t", encoding=encoding)
                missing = required_columns.difference(raw_df.columns)
                if missing:
                    raise ValueError(f"Missing columns: {sorted(missing)}")

                dates = pd.to_datetime(raw_df["Time"], errors="coerce")
                print(f"Dataset path: {path.resolve()}")
                print(f"Encoding: {encoding}")
                print(f"Rows: {len(raw_df):,}")
                print(f"Dates: {dates.min().date()} -> {dates.max().date()}")
                print(f"Columns: {list(raw_df.columns)}")
                return raw_df
            except Exception as exc:
                load_errors.append(f"{path} ({encoding}): {exc}")

    searched = "\n".join(str(path) for path in candidate_paths)
    details = "\n".join(load_errors)
    raise FileNotFoundError(
        f"Dataset not found. Searched:\n{searched}\n\nLoad errors:\n{details}"
    )


def prepare_btc_data(raw_df, start_date=START_DATE):
    """Clean raw BTC data and build the weekly log-price dataset used by the model."""
    df = raw_df.rename(columns={
        "Time": "date",
        "BTC / USD Denominated Closing Price": "price",
    }).copy()

    df["date"] = pd.to_datetime(df["date"])
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = df.dropna(subset=["date", "price"])
    df = df.sort_values("date")

    btc = df[df["date"] >= start_date].copy()
    weekly = (
        btc.set_index("date")
           .resample("W")
           .last()
           .dropna()
           .reset_index()
    )
    weekly["log_price"] = np.log(weekly["price"])

    return df, btc, weekly


def get_turning_point(weekly, start, end, kind, label, phase):
    """Find the real weekly max/min inside a manual cycle window."""
    subset = weekly[(weekly["date"] >= start) & (weekly["date"] <= end)].copy()
    if len(subset) == 0:
        raise ValueError(f"No data between {start} and {end}")

    if kind == "max":
        row = subset.loc[subset["price"].idxmax()]
    else:
        row = subset.loc[subset["price"].idxmin()]

    return {
        "date": row["date"],
        "price": row["price"],
        "log_price": np.log(row["price"]),
        "label": label,
        "phase": phase,
        "kind": kind,
    }


def build_turning_points(weekly):
    """Build the manual cycle anchors used to map log-price into cycle phase."""
    windows = [
        ("2013-01-01", "2013-04-30", "min", "Inicio 2013", -0.35),
        ("2013-10-01", "2014-01-31", "max", "Max 2013", 1.00),
        ("2014-12-01", "2015-03-31", "min", "Min 2015", -0.75),
        ("2017-11-01", "2018-01-31", "max", "Max 2017", 0.75),
        ("2018-11-01", "2019-02-28", "min", "Min 2018", -0.50),
        ("2021-10-01", "2021-12-31", "max", "Max 2021", 0.50),
        ("2022-10-01", "2023-01-31", "min", "Min 2022", -0.35),
        ("2025-01-01", "2026-03-01", "max", "Max 2025", 0.35),
    ]

    turns = [
        get_turning_point(weekly, start, end, kind, label, phase)
        for start, end, kind, label, phase in windows
    ]

    return pd.DataFrame(turns).sort_values("date").reset_index(drop=True)


def compute_damping_parameters(turning_points):
    """Estimate exponential damping bands from historical peak/trough amplitudes."""
    peaks = turning_points[turning_points["kind"] == "max"].copy()
    troughs = turning_points[
        (turning_points["kind"] == "min") &
        (turning_points["label"] != "Inicio 2013")
    ].copy()

    peak_dates = peaks["date"].to_list()
    peak_amps = peaks["phase"].abs().to_numpy()
    trough_dates = troughs["date"].to_list()
    trough_amps = troughs["phase"].abs().to_numpy()

    peak_betas = []
    for i in range(len(peak_amps) - 1):
        dt = year_diff(peak_dates[i + 1], peak_dates[i])
        peak_betas.append(-np.log(peak_amps[i + 1] / peak_amps[i]) / dt)

    trough_betas = []
    for i in range(len(trough_amps) - 1):
        dt = year_diff(trough_dates[i + 1], trough_dates[i])
        trough_betas.append(-np.log(trough_amps[i + 1] / trough_amps[i]) / dt)

    peak_betas = np.array(peak_betas)
    trough_betas = np.array(trough_betas)

    beta_peak_mean = peak_betas.mean()
    beta_peak_std = peak_betas.std(ddof=1)
    beta_trough_mean = trough_betas.mean()
    beta_trough_std = trough_betas.std(ddof=1)

    return {
        "peaks": peaks,
        "troughs": troughs,
        "peak_betas": peak_betas,
        "trough_betas": trough_betas,
        "beta_peak_mean": beta_peak_mean,
        "beta_peak_std": beta_peak_std,
        "beta_peak_low": max(0.001, beta_peak_mean - beta_peak_std),
        "beta_peak_high": beta_peak_mean + beta_peak_std,
        "beta_trough_mean": beta_trough_mean,
        "beta_trough_std": beta_trough_std,
        "beta_trough_low": max(0.001, beta_trough_mean - beta_trough_std),
        "beta_trough_high": beta_trough_mean + beta_trough_std,
    }


def estimate_next_cycle_windows(
    turning_points,
    damping,
    drawdown_suave=DRAWDOWN_SUAVE,
    drawdown_duro=DRAWDOWN_DURO,
    mult_min=MULT_MIN,
    mult_max=MULT_MAX,
):
    """Estimate next trough/peak timing windows and price ranges."""
    peaks = damping["peaks"]
    troughs = damping["troughs"]
    peak_dates = peaks["date"].to_list()

    peak_to_peak = np.array([
        year_diff(peak_dates[i + 1], peak_dates[i])
        for i in range(len(peak_dates) - 1)
    ])

    historical_peak_to_trough = np.array([
        year_diff(troughs.iloc[0]["date"], peaks.iloc[0]["date"]),
        year_diff(troughs.iloc[1]["date"], peaks.iloc[1]["date"]),
        year_diff(troughs.iloc[2]["date"], peaks.iloc[2]["date"]),
    ])

    pp_mean = peak_to_peak.mean()
    pp_std = peak_to_peak.std(ddof=1)
    pt_mean = historical_peak_to_trough.mean()
    pt_std = historical_peak_to_trough.std(ddof=1)

    last_peak = peaks.iloc[-1]
    last_peak_date = last_peak["date"]
    last_peak_price = last_peak["price"]

    next_min_early = last_peak_date + pd.to_timedelta((pt_mean - pt_std) * 365.25, unit="D")
    next_min_late = last_peak_date + pd.to_timedelta((pt_mean + pt_std) * 365.25, unit="D")
    next_min_center = last_peak_date + pd.to_timedelta(pt_mean * 365.25, unit="D")

    next_max_early = last_peak_date + pd.to_timedelta((pp_mean - pp_std) * 365.25, unit="D")
    next_max_late = last_peak_date + pd.to_timedelta((pp_mean + pp_std) * 365.25, unit="D")
    next_max_center = last_peak_date + pd.to_timedelta(pp_mean * 365.25, unit="D")

    min_price_low = last_peak_price * (1 - drawdown_duro)
    min_price_high = last_peak_price * (1 - drawdown_suave)
    max_price_low = min_price_low * mult_min
    max_price_high = min_price_high * mult_max

    last_trough = troughs.iloc[-1]
    dt_next_trough = year_diff(next_min_center, last_trough["date"])

    next_trough_amp_less_damp = abs(last_trough["phase"]) * np.exp(
        -damping["beta_trough_low"] * dt_next_trough
    )
    next_trough_amp_more_damp = abs(last_trough["phase"]) * np.exp(
        -damping["beta_trough_high"] * dt_next_trough
    )
    next_trough_phase = -np.mean([
        next_trough_amp_less_damp,
        next_trough_amp_more_damp,
    ])

    projected_min = {
        "date": next_min_center,
        "price": (min_price_low + min_price_high) / 2,
        "log_price": np.log((min_price_low + min_price_high) / 2),
        "label": "Min proyectado",
        "phase": next_trough_phase,
        "kind": "min",
    }

    turns_for_mapping = pd.concat([
        turning_points,
        pd.DataFrame([projected_min]),
    ]).sort_values("date").reset_index(drop=True)

    return {
        "peak_to_peak": peak_to_peak,
        "historical_peak_to_trough": historical_peak_to_trough,
        "pp_mean": pp_mean,
        "pp_std": pp_std,
        "pt_mean": pt_mean,
        "pt_std": pt_std,
        "last_peak": last_peak,
        "next_min_early": next_min_early,
        "next_min_late": next_min_late,
        "next_min_center": next_min_center,
        "next_max_early": next_max_early,
        "next_max_late": next_max_late,
        "next_max_center": next_max_center,
        "min_price_low": min_price_low,
        "min_price_high": min_price_high,
        "max_price_low": max_price_low,
        "max_price_high": max_price_high,
        "projected_min": projected_min,
        "turns_for_mapping": turns_for_mapping,
    }


def build_cycle_oscillator(weekly, turns_for_mapping, rolling_smooth=ROLLING_SMOOTH):
    """Map weekly log-price into cycle phase, normalized segment by segment."""
    segments = []

    for i in range(len(turns_for_mapping) - 1):
        start = turns_for_mapping.iloc[i]
        end = turns_for_mapping.iloc[i + 1]
        seg_end_date = min(end["date"], weekly["date"].max())

        seg = weekly[
            (weekly["date"] >= start["date"]) &
            (weekly["date"] <= seg_end_date)
        ].copy()

        if len(seg) == 0:
            continue

        log_start = start["log_price"]
        log_end = end["log_price"]
        phase_start = start["phase"]
        phase_end = end["phase"]

        if abs(log_end - log_start) < 1e-9:
            continue

        seg["ratio"] = (seg["log_price"] - log_start) / (log_end - log_start)
        seg["cycle_raw"] = phase_start + seg["ratio"] * (phase_end - phase_start)

        low = min(phase_start, phase_end) - 0.12
        high = max(phase_start, phase_end) + 0.12
        seg["cycle_raw"] = seg["cycle_raw"].clip(low, high)

        segments.append(seg)

    cycle = pd.concat(segments).sort_values("date").drop_duplicates("date")
    cycle["cycle_smooth"] = (
        cycle["cycle_raw"]
        .rolling(rolling_smooth, center=True, min_periods=1)
        .mean()
    )

    return cycle


def compute_envelopes(damping, future_end=FUTURE_END, n_points=700):
    """Compute upper/lower exponential damping envelopes for the oscillator."""
    peaks = damping["peaks"]
    troughs = damping["troughs"]

    env_dates = pd.date_range(peaks.iloc[0]["date"], future_end, periods=n_points)

    upper_anchor_date = peaks.iloc[0]["date"]
    upper_anchor_amp = abs(peaks.iloc[0]["phase"])
    upper_t = np.array([
        max(0, year_diff(date, upper_anchor_date))
        for date in env_dates
    ])
    upper_less_damp = upper_anchor_amp * np.exp(-damping["beta_peak_low"] * upper_t)
    upper_more_damp = upper_anchor_amp * np.exp(-damping["beta_peak_high"] * upper_t)

    lower_anchor_date = troughs.iloc[0]["date"]
    lower_anchor_amp = abs(troughs.iloc[0]["phase"])
    lower_t = np.array([
        max(0, year_diff(date, lower_anchor_date))
        for date in env_dates
    ])
    lower_less_damp = -lower_anchor_amp * np.exp(-damping["beta_trough_low"] * lower_t)
    lower_more_damp = -lower_anchor_amp * np.exp(-damping["beta_trough_high"] * lower_t)

    return {
        "env_dates": env_dates,
        "upper_less_damp": upper_less_damp,
        "upper_more_damp": upper_more_damp,
        "lower_less_damp": lower_less_damp,
        "lower_more_damp": lower_more_damp,
    }


def plot_cycle_oscillator(
    weekly,
    cycle,
    turning_points,
    envelopes,
    windows,
    future_end=FUTURE_END,
):
    """Plot the BTC cycle oscillator with cycle backgrounds, envelopes, and windows."""
    fig, ax = plt.subplots(figsize=(18, 8))

    cycle_spans = [
        (pd.Timestamp("2013-11-01"), pd.Timestamp("2017-12-31"), "#eaf4ff", "Ciclo 2013-2017"),
        (pd.Timestamp("2017-12-31"), pd.Timestamp("2021-11-30"), "#fff3e6", "Ciclo 2017-2021"),
        (pd.Timestamp("2021-11-30"), pd.Timestamp("2025-12-31"), "#edf8ee", "Ciclo 2021-2025"),
        (pd.Timestamp("2025-12-31"), future_end, "#f5edff", "Ciclo proyectado 2025-2029"),
    ]

    for start, end, color, _ in cycle_spans:
        ax.axvspan(start, end, color=color, alpha=0.45, zorder=0)

    ax.plot(
        cycle["date"],
        cycle["cycle_smooth"],
        linewidth=2.8,
        color="#3f73b5",
        label="Ciclo BTC normalizado con datos reales",
        zorder=4,
    )
    ax.plot(
        cycle["date"],
        cycle["cycle_raw"],
        linewidth=0.7,
        alpha=0.25,
        color="#f2a65a",
        label="Detalle semanal real",
        zorder=3,
    )
    ax.axhline(
        0,
        linewidth=2.8,
        color="#3f73b5",
        alpha=0.9,
        label="Tendencia normalizada",
        zorder=2,
    )

    env_dates = envelopes["env_dates"]
    ax.fill_between(
        env_dates,
        envelopes["upper_more_damp"],
        envelopes["upper_less_damp"],
        color="#ff7b72",
        alpha=0.16,
        label="Envolvente superior +/- amortiguamiento",
        zorder=1,
    )
    ax.plot(env_dates, envelopes["upper_less_damp"], "--", color="#ff7b72", linewidth=2, zorder=5)
    ax.plot(env_dates, envelopes["upper_more_damp"], "--", color="#ff7b72", linewidth=2, zorder=5)

    ax.fill_between(
        env_dates,
        envelopes["lower_less_damp"],
        envelopes["lower_more_damp"],
        color="#2fbf71",
        alpha=0.16,
        label="Envolvente inferior +/- amortiguamiento",
        zorder=1,
    )
    ax.plot(env_dates, envelopes["lower_less_damp"], "--", color="#2fbf71", linewidth=2, zorder=5)
    ax.plot(env_dates, envelopes["lower_more_damp"], "--", color="#2fbf71", linewidth=2, zorder=5)

    ax.axvspan(
        windows["next_min_early"],
        windows["next_min_late"],
        color="#f6bd60",
        alpha=0.22,
        label="Ventana proximo minimo",
        zorder=1,
    )
    ax.axvline(windows["next_min_center"], color="#f6bd60", linestyle=":", linewidth=3, zorder=6)

    ax.axvspan(
        windows["next_max_early"],
        windows["next_max_late"],
        color="#c77dff",
        alpha=0.18,
        label="Ventana proximo maximo",
        zorder=1,
    )
    ax.axvline(windows["next_max_center"], color="#c77dff", linestyle=":", linewidth=3, zorder=6)

    ax.scatter(
        turning_points["date"],
        turning_points["phase"],
        s=70,
        color="#3f73b5",
        zorder=7,
    )

    for _, row in turning_points.iterrows():
        text = f"{row['label']}\n${row['price']:,.0f}"
        offset = 12 if row["phase"] >= 0 else -28
        va = "bottom" if row["phase"] >= 0 else "top"
        ax.annotate(
            text,
            (row["date"], row["phase"]),
            textcoords="offset points",
            xytext=(0, offset),
            ha="center",
            va=va,
            fontsize=8.5,
            zorder=8,
        )

    last = weekly.iloc[-1]
    last_cycle = cycle.iloc[-1]
    ax.scatter(
        last_cycle["date"],
        last_cycle["cycle_smooth"],
        s=90,
        color="#f28e2b",
        zorder=8,
    )
    ax.annotate(
        f"Hoy\n${last['price']:,.0f}",
        (last_cycle["date"], last_cycle["cycle_smooth"]),
        textcoords="offset points",
        xytext=(12, -8),
        ha="left",
        va="center",
        fontsize=9,
        zorder=9,
    )

    min_text = (
        "Proximo minimo\n"
        f"{windows['next_min_early'].strftime('%b %Y')} - {windows['next_min_late'].strftime('%b %Y')}\n"
        f"${windows['min_price_low']:,.0f} - ${windows['min_price_high']:,.0f}"
    )
    max_text = (
        "Proximo maximo\n"
        f"{windows['next_max_early'].strftime('%b %Y')} - {windows['next_max_late'].strftime('%b %Y')}\n"
        f"${windows['max_price_low']:,.0f} - ${windows['max_price_high']:,.0f}"
    )

    ax.text(
        windows["next_min_center"],
        -0.92,
        min_text,
        ha="center",
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#f6bd60", alpha=0.96),
        zorder=10,
    )
    ax.text(
        windows["next_max_center"],
        0.82,
        max_text,
        ha="center",
        va="bottom",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#c77dff", alpha=0.96),
        zorder=10,
    )

    cycle_labels = [
        (pd.Timestamp("2015-12-01"), "Ciclo 2013-2017"),
        (pd.Timestamp("2019-11-01"), "Ciclo 2017-2021"),
        (pd.Timestamp("2023-11-01"), "Ciclo 2021-2025"),
        (pd.Timestamp("2028-02-01"), "Ciclo proyectado"),
    ]
    for x_pos, label in cycle_labels:
        ax.text(x_pos, -1.08, label, ha="center", va="bottom", fontsize=9, alpha=0.65)

    ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    ax.set_yticklabels([
        "Capitulacion",
        "Infravalorado",
        "Tendencia",
        "Sobrecalentado",
        "Euforia",
    ])
    ax.set_ylim(-1.15, 1.15)
    ax.set_xlim(pd.Timestamp("2013-01-01"), future_end)
    ax.set_title(
        "Bitcoin: oscilador de ciclos desde 2013 con detalle semanal real\n"
        "y envolventes exponenciales de amortiguamiento",
        fontsize=15,
    )
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Fase del ciclo")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9, frameon=True)

    plt.tight_layout(rect=[0, 0, 0.82, 1])
    return fig, ax


## Ejecutar modelo y grafico principal

In [ ]:
raw_df = load_btc_data()
df, btc, weekly = prepare_btc_data(raw_df)

turns_actual = build_turning_points(weekly)
damping = compute_damping_parameters(turns_actual)
windows = estimate_next_cycle_windows(turns_actual, damping)
cycle = build_cycle_oscillator(weekly, windows["turns_for_mapping"])
envelopes = compute_envelopes(damping)

fig, ax = plot_cycle_oscillator(
    weekly=weekly,
    cycle=cycle,
    turning_points=turns_actual,
    envelopes=envelopes,
    windows=windows,
)

plt.show()


## Resumen numerico

In [ ]:
latest = df.iloc[-1]

print("Resumen numerico:")
print(f"Fecha/precio actual: {latest['date'].date()} - ${latest['price']:,.0f}")
print()
print("Betas de amortiguamiento:")
print(f"Beta superior menos amortiguamiento: {damping['beta_peak_low']:.4f} / año")
print(f"Beta superior mas amortiguamiento:   {damping['beta_peak_high']:.4f} / año")
print(f"Beta inferior menos amortiguamiento: {damping['beta_trough_low']:.4f} / año")
print(f"Beta inferior mas amortiguamiento:   {damping['beta_trough_high']:.4f} / año")
print()
print(f"Periodo pico->pico: {windows['pp_mean']:.2f} +/- {windows['pp_std']:.2f} años")
print(f"Tiempo pico->minimo: {windows['pt_mean']:.2f} +/- {windows['pt_std']:.2f} años")
print()
print("Rangos estimados:")
print(
    "Proximo minimo: "
    f"{windows['next_min_early'].strftime('%Y-%m-%d')} a "
    f"{windows['next_min_late'].strftime('%Y-%m-%d')} | "
    f"${windows['min_price_low']:,.0f} - ${windows['min_price_high']:,.0f}"
)
print(
    "Proximo maximo: "
    f"{windows['next_max_early'].strftime('%Y-%m-%d')} a "
    f"{windows['next_max_late'].strftime('%Y-%m-%d')} | "
    f"${windows['max_price_low']:,.0f} - ${windows['max_price_high']:,.0f}"
)
